
# Example 2 — Nonlinear cylinder target in \(d=2\): collective band steering

This notebook implements **Algorithm 2** / the **nonlinear cylinder terminal condition** with a
single smooth periodic observable instead of the symmetric two-region split.

We use the scalar observable
\[
\phi(x,y)=\exp\!\big(\kappa(\cos(2\pi(x-c))-1)\big),
\]
which is a smooth **vertical target band** on the flat torus \([0,1)^2\).
The terminal functional asks that the weighted band score
\[
\mu(\phi)=\sum_{i=1}^n s_i \phi(x_i)
\]
should be close to a prescribed scalar target \(a\).

Why this example tends to be much more stable than the two-region split:
- it uses **\(K=1\)**, so the quadrature is only one-dimensional
- the target is **asymmetric** (one band, not two competing wells)
- the band is **broad**, so particles do not need to hit a tiny region
- the conditioning is still genuinely **collective/unlabelled**: only the total weighted band score matters, not which particle gets there

The plotting/animation follows the same Plotly slider style as the provided
house \(\to\) flower notebook.


In [ ]:
import numpy as np
import sys
from pathlib import Path
from numpy.polynomial.hermite import hermgauss


ROOT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents, Path("/mnt/data").resolve())
ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "wasserstein_conditioning_algorithms.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate wasserstein_conditioning_algorithms.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.wasserstein_conditioning_algorithms import simulate_nonlinear_cylinder_quadrature_em

np.set_printoptions(precision=3, suppress=True)


In [ ]:
import plotly.graph_objects as go

from notebooks.support import (
    center_trace,
    configure_plotly,
    line_trace as _line_trace,
    make_particle_animation as _make_particle_animation,
)

configure_plotly()


def line_trace(points, name, color="rgba(80,80,80,0.55)", dash="dot", close=False, showlegend=True, marker_size=6):
    return _line_trace(
        points,
        name,
        color=color,
        dash=dash,
        close=close,
        showlegend=showlegend,
        marker_size=marker_size,
        mode="lines+markers",
    )



def make_particle_animation(
    positions,
    times,
    masses,
    title,
    static_traces=None,
    marker_size=18,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
):
    return _make_particle_animation(
        positions,
        times,
        masses,
        title,
        static_traces=static_traces,
        marker_size=marker_size,
        x_range=x_range,
        y_range=y_range,
        mass_format=".3f",
        time_formatter=lambda t, h: f"time = {t:.6f}",
        slider_label_formatter=lambda t, h: f"{t:.6f}",
        currentvalue_prefix="time = ",
        width=800,
        height=700,
        play_frame_duration=110,
    )



def rectangle_outline(x0, x1, y0=0.0, y1=1.0):
    return np.array([
        [x0, y0],
        [x1, y0],
        [x1, y1],
        [x0, y1],
    ], dtype=float)


In [ ]:

def smooth_vertical_band(center_x=0.0, sharpness=1.2):
    """Smooth periodic band depending only on x."""
    def phi(points):
        pts = np.asarray(points, dtype=float)
        return np.exp(sharpness * (np.cos(2.0 * np.pi * (pts[..., 0] - center_x)) - 1.0))
    return phi

def gauss_hermite_cylinder_quadrature(lambda_, order):
    """Quadrature for (2πλ)^(-1/2) exp(-η^2 / (2λ)) dη."""
    nodes, weights = hermgauss(order)
    eta = np.sqrt(2.0 * lambda_) * nodes[:, None]
    quad_weights = weights / np.sqrt(np.pi)
    return eta, quad_weights

def initial_vertical_strip(masses, x_center=0.48, y_low=0.16, y_high=0.84, x_jitter=0.012):
    n = len(masses)
    ys = np.linspace(y_low, y_high, n)
    xs = x_center + x_jitter * np.sin(np.linspace(0.0, 2.0 * np.pi, n, endpoint=False))
    return np.mod(np.stack([xs, ys], axis=1), 1.0)

def weighted_band_score(positions, masses, observable):
    return np.array([np.sum(masses * observable(pos)) for pos in positions], dtype=float)

def band_half_height_width(sharpness):
    # width defined by phi = 1/2, used only for the plotting guide
    arg = 1.0 + np.log(0.5) / sharpness
    arg = np.clip(arg, -1.0, 1.0)
    return np.arccos(arg) / (2.0 * np.pi)

# --- masses and geometric setup ---
masses = np.array([0.30, 0.24, 0.18, 0.12, 0.09, 0.07], dtype=float)
masses = masses / masses.sum()

band_center_x = 0.0
band_sharpness = 1.2
target_band_score = 0.18

initial_positions = initial_vertical_strip(masses, x_center=0.48, y_low=0.16, y_high=0.84, x_jitter=0.012)
observable = smooth_vertical_band(center_x=band_center_x, sharpness=band_sharpness)

lambda_ = 200.0
horizon = 0.002
steps = 180
step_size = horizon / steps
quadrature_order = 13
grid_shape = 64
seed = 2

quadrature_nodes, quadrature_weights = gauss_hermite_cylinder_quadrature(lambda_, quadrature_order)

half_width = band_half_height_width(band_sharpness)
left_band = rectangle_outline(0.0, half_width, 0.0, 1.0)
right_band = rectangle_outline(1.0 - half_width, 1.0, 0.0, 1.0)
initial_strip = rectangle_outline(0.48 - 0.04, 0.48 + 0.04, 0.0, 1.0)

print("masses:", masses)
print("target band score a:", target_band_score)
print("band half-height width used for the guide:", float(half_width))
print("lambda:", lambda_, "horizon:", horizon, "step size:", step_size)
print("quadrature order:", quadrature_order, "grid shape:", grid_shape, "seed:", seed)


In [ ]:

def run_band_simulation(seed, *, store_drifts=True):
    rng = np.random.default_rng(seed)
    return simulate_nonlinear_cylinder_quadrature_em(
        masses=masses,
        observables=[observable],
        target_vector=np.array([target_band_score], dtype=float),
        lambda_=lambda_,
        horizon=horizon,
        step_size=step_size,
        initial_positions=initial_positions,
        quadrature_nodes=quadrature_nodes,
        quadrature_weights=quadrature_weights,
        grid_shape=grid_shape,
        rng=rng,
        store_drifts=store_drifts,
    )

sim = run_band_simulation(seed, store_drifts=True)
band_score = weighted_band_score(sim.positions, sim.masses, observable)

print("positions array shape:", sim.positions.shape)
print("final time:", float(sim.times[-1]))
print("initial weighted band score:", float(band_score[0]))
print("final weighted band score:", float(band_score[-1]))


In [ ]:

static_traces = [
    line_trace(initial_strip, name="initial strip guide", color="rgba(30, 144, 255, 0.45)", dash="dash", close=True),
    line_trace(left_band, name="target band guide", color="rgba(220, 20, 60, 0.55)", dash="dot", close=True),
    line_trace(right_band, name="target band guide (wrap copy)", color="rgba(220, 20, 60, 0.55)", dash="dot", close=True, showlegend=False),
]

fig = make_particle_animation(
    positions=sim.positions,
    times=sim.times,
    masses=sim.masses,
    title="Example 2: nonlinear cylinder target (collective band steering)",
    static_traces=static_traces,
    marker_size=20,
)
fig.show()


In [ ]:

band_fig = go.Figure()
band_fig.add_trace(
    go.Scatter(
        x=sim.times,
        y=band_score,
        mode="lines",
        name="weighted band score",
    )
)
band_fig.add_hline(
    y=target_band_score,
    line_dash="dash",
    annotation_text="target a",
    annotation_position="top left",
)
band_fig.update_layout(
    title="Collective observable over time",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="weighted band score",
)
band_fig.show()

print("initial score:", float(band_score[0]))
print("final score:", float(band_score[-1]))
print("target a:", float(target_band_score))



### Seed robustness check

The next cell reruns the **same** setup for a small list of seeds and overlays the trajectories of the
same collective observable \(\mu_t(\phi)\).
For this example, the terminal band score is fairly concentrated across seeds, even though the
*identity* of the particles that enter the band can vary from run to run.


In [ ]:

robustness_seeds = [0, 1, 2, 3, 4]
seed_to_score = {}

for s in robustness_seeds:
    sim_s = run_band_simulation(s, store_drifts=False)
    seed_to_score[s] = weighted_band_score(sim_s.positions, sim_s.masses, observable)

final_scores = np.array([seed_to_score[s][-1] for s in robustness_seeds], dtype=float)

robust_fig = go.Figure()
for s in robustness_seeds:
    robust_fig.add_trace(
        go.Scatter(
            x=sim.times,
            y=seed_to_score[s],
            mode="lines",
            name=f"seed {s}",
            opacity=0.75,
        )
    )

robust_fig.add_hline(
    y=target_band_score,
    line_dash="dash",
    annotation_text="target a",
    annotation_position="top left",
)

robust_fig.update_layout(
    title="Seed robustness: weighted band score across several runs",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="weighted band score",
)
robust_fig.show()

print("final weighted band scores by seed:")
for s in robustness_seeds:
    print(f"  seed {s}: {float(seed_to_score[s][-1]):.6f}")

print()
print("mean final score:", float(final_scores.mean()))
print("std of final score:", float(final_scores.std()))
